# 10. YOLO11-seg - river plume segmentation

Instance-segmentation baseline based on Ultralytics **YOLO11l-seg** with
COCO-pretrained weights. The dataset must be in the YOLO segmentation format
(polygon labels) described by `data.yaml`; the images/masks are the same
`split_data` splits used by the other models.

Hyperparameters below reproduce the final training run of the study. They were
selected with a grid search over `lr0`, `weight_decay`, `momentum`, `dropout`
and rotation `degrees` (AdamW, short 15-25-epoch runs scored on the validation
split).

In [ ]:
%pip install -q "ultralytics==8.3.40"

## 1. Configuration

In [ ]:
import os
import random

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ============================================================================
# CONFIGURATION - final training run of the study
# ============================================================================

DATA_YAML = "data.yaml"          # YOLO dataset description (train/val/test paths)
BASE_WEIGHTS = "yolo11l-seg.pt"  # COCO-pretrained checkpoint

EPOCHS = 200
PATIENCE = 30                    # early-stopping patience (epochs)
BATCH = 8
IMGSZ = 640                      # Ultralytics default input size

OPTIMIZER = "AdamW"
LR0 = 0.0001                     # initial learning rate (grid-search optimum)
WEIGHT_DECAY = 0.01
MOMENTUM = 0.9
DROPOUT = 0.1
DEGREES = 180                    # rotation augmentation range
FREEZE = 0                       # no frozen layers in the final run
WARMUP_EPOCHS = 3
COS_LR = True                    # cosine learning-rate schedule
AMP = True                       # mixed precision

PROJECT = "yolo11_river_plumes"
RUN_NAME = "final"

# Test data for the canonical evaluation below.
TEST_IMG_DIR = "split_data/test/images"
TEST_MASK_DIR = "split_data/test/Masks"
PRED_THRESHOLD = 0.5

PROBABILITY_EXPORT_DIR = "probability_masks/test/yolo"
BINARY_EXPORT_DIR = "test_masks/yolo"

## 2. Training

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_WEIGHTS)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    patience=PATIENCE,
    lr0=LR0,
    weight_decay=WEIGHT_DECAY,
    momentum=MOMENTUM,
    dropout=DROPOUT,
    degrees=DEGREES,
    freeze=FREEZE,
    warmup_epochs=WARMUP_EPOCHS,
    optimizer=OPTIMIZER,
    cos_lr=COS_LR,
    batch=BATCH,
    imgsz=IMGSZ,
    amp=AMP,
    device=0,
    project=PROJECT,
    name=RUN_NAME,
    save=True,
    save_period=30,
    exist_ok=True,
)

print("Training finished.")
print(f"Best weights: {PROJECT}/{RUN_NAME}/weights/best.pt")

## 3. Validation metrics (Ultralytics)

In [ ]:
best_model = YOLO(f"{PROJECT}/{RUN_NAME}/weights/best.pt")
val_metrics = best_model.val(data=DATA_YAML)
print(val_metrics)

## 4. Test-set evaluation (canonical metrics)

Per-instance YOLO masks are merged into a single semantic probability map
(pixel-wise maximum over instances) and evaluated with the same metric
functions as the other models: Dice, Accuracy, Precision, Recall and Hausdorff
distances at the native ground-truth resolution.

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
from scipy.ndimage import distance_transform_edt


def compute_dice(mask_true, mask_pred, eps=1e-7):
    """Dice coefficient between two binary masks (numpy)."""
    mask_true = mask_true.astype(np.float32)
    mask_pred = mask_pred.astype(np.float32)

    intersection = np.sum(mask_true * mask_pred)
    total = np.sum(mask_true) + np.sum(mask_pred)

    if total == 0:
        # Both masks empty: perfect agreement.
        return 1.0

    return (2.0 * intersection) / (total + eps)


def compute_metrics(mask_true, mask_pred, eps=1e-7):
    """Pixel-wise Accuracy, Precision and Recall for binary segmentation."""
    y_true = mask_true.flatten().astype(np.uint8)
    y_pred = mask_pred.flatten().astype(np.uint8)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    TN = np.sum((y_true == 0) & (y_pred == 0))

    total = TP + TN + FP + FN
    accuracy = (TP + TN) / (total + eps)
    precision = 0.0 if (TP + FP) == 0 else TP / (TP + FP)
    recall = 0.0 if (TP + FN) == 0 else TP / (TP + FN)

    return accuracy, precision, recall


def compute_hausdorff_distances(mask_true, mask_pred):
    """HD95, Average HD and Max HD between two binary masks.

    Distances are computed symmetrically with Euclidean distance transforms
    over the full masks of an image.
    Returns (hd95, avg_hd, max_hd) in pixels.
    """
    true_coords = np.argwhere(mask_true > 0)
    pred_coords = np.argwhere(mask_pred > 0)

    if len(true_coords) == 0 or len(pred_coords) == 0:
        if len(true_coords) == 0 and len(pred_coords) == 0:
            return 0.0, 0.0, 0.0
        # One mask is empty: report the image diagonal as the distance.
        max_dist = np.sqrt(mask_true.shape[0] ** 2 + mask_true.shape[1] ** 2)
        return max_dist, max_dist, max_dist

    distances_pred_to_true = distance_transform_edt(1 - mask_true)
    distances_pred_to_true = distances_pred_to_true[pred_coords[:, 0], pred_coords[:, 1]]

    distances_true_to_pred = distance_transform_edt(1 - mask_pred)
    distances_true_to_pred = distances_true_to_pred[true_coords[:, 0], true_coords[:, 1]]

    all_distances = np.concatenate([distances_pred_to_true, distances_true_to_pred])

    hd95 = np.percentile(all_distances, 95)
    avg_hd = np.mean(all_distances)
    max_hd = np.max(all_distances)

    return hd95, avg_hd, max_hd


def binarize_mask(mask, threshold=0.5):
    """Binarize a mask with the given threshold."""
    return (mask > threshold).astype(np.uint8)


def find_corresponding_mask(img_name, mask_files):
    """Find the ground-truth mask file matching an image file name."""
    img_stem = os.path.splitext(img_name)[0]

    possible_names = [
        img_name,
        f"{img_stem}.png",
        f"{img_stem}.jpg",
        f"{img_stem}.jpeg",
        f"{img_stem}_mask.png",
        f"mask_{img_stem}.png",
        f"{img_stem}_gt.png",
        f"{img_stem}_groundtruth.png",
    ]

    for possible_name in possible_names:
        if possible_name in mask_files:
            return possible_name
    return None


def resize_to_match(mask, target_size):
    """Resize a binary mask to target_size with nearest-neighbour resampling."""
    if isinstance(mask, np.ndarray):
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
    else:
        mask_pil = mask

    resized = mask_pil.resize(target_size, resample=Image.NEAREST)
    return (np.array(resized) > 127).astype(np.uint8)

In [ ]:
import torch


def predict_probability_mask(model, img_path, imgsz=IMGSZ):
    """Run YOLO11-seg on one image and merge instance masks.

    Returns a float32 probability map in [0, 1] at the network mask
    resolution (pixel-wise maximum over all predicted instances), or None if
    nothing was detected.
    """
    results = model.predict(img_path, imgsz=imgsz, verbose=False)
    r = results[0]
    if r.masks is None or r.masks.data.shape[0] == 0:
        return None
    masks = r.masks.data  # (num_instances, H, W), float in [0, 1]
    prob = torch.max(masks, dim=0).values.cpu().numpy().astype(np.float32)
    return prob


img_names = sorted([f for f in os.listdir(TEST_IMG_DIR)
                    if f.lower().endswith((".png", ".jpg", ".jpeg"))])
mask_files = sorted([f for f in os.listdir(TEST_MASK_DIR)
                     if f.lower().endswith((".png", ".jpg", ".jpeg"))])

dice_scores, accuracy_scores = [], []
precision_scores, recall_scores = [], []
hd95_scores, avg_hd_scores, max_hd_scores = [], [], []

for i, img_name in enumerate(img_names):
    mask_name = find_corresponding_mask(img_name, mask_files)
    if mask_name is None:
        print(f"Mask not found for {img_name}, skipping")
        continue

    true_mask_img = Image.open(os.path.join(TEST_MASK_DIR, mask_name)).convert("L")
    true_mask_bin = (np.array(true_mask_img) > 127).astype(np.uint8)

    prob = predict_probability_mask(best_model, os.path.join(TEST_IMG_DIR, img_name))
    if prob is None:
        pred_bin = np.zeros_like(true_mask_bin, dtype=np.uint8)
    else:
        pred_bin = binarize_mask(prob, threshold=PRED_THRESHOLD)
        pred_bin = resize_to_match(pred_bin, true_mask_img.size)

    if true_mask_bin.shape != pred_bin.shape:
        min_h = min(true_mask_bin.shape[0], pred_bin.shape[0])
        min_w = min(true_mask_bin.shape[1], pred_bin.shape[1])
        true_mask_bin = true_mask_bin[:min_h, :min_w]
        pred_bin = pred_bin[:min_h, :min_w]

    dice_scores.append(compute_dice(true_mask_bin, pred_bin))
    acc, prec, rec = compute_metrics(true_mask_bin, pred_bin)
    accuracy_scores.append(acc)
    precision_scores.append(prec)
    recall_scores.append(rec)
    hd95, avg_hd, max_hd = compute_hausdorff_distances(true_mask_bin, pred_bin)
    hd95_scores.append(hd95)
    avg_hd_scores.append(avg_hd)
    max_hd_scores.append(max_hd)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(img_names)}")

print("\n" + "=" * 70)
print("TEST-SET RESULTS: YOLO11-seg")
print("=" * 70)
print(f"Mean Dice Coefficient: {np.mean(dice_scores):.4f} +/- {np.std(dice_scores):.4f}")
print(f"Mean Accuracy:         {np.mean(accuracy_scores):.4f} +/- {np.std(accuracy_scores):.4f}")
print(f"Mean Precision:        {np.mean(precision_scores):.4f} +/- {np.std(precision_scores):.4f}")
print(f"Mean Recall:           {np.mean(recall_scores):.4f} +/- {np.std(recall_scores):.4f}")
print(f"Mean HD95:             {np.mean(hd95_scores):.4f} +/- {np.std(hd95_scores):.4f}")
print(f"Mean Average HD:       {np.mean(avg_hd_scores):.4f} +/- {np.std(avg_hd_scores):.4f}")
print(f"Mean Max HD:           {np.mean(max_hd_scores):.4f} +/- {np.std(max_hd_scores):.4f}")

## 5. Export predictions for the ensemble

In [ ]:
os.makedirs(PROBABILITY_EXPORT_DIR, exist_ok=True)
os.makedirs(BINARY_EXPORT_DIR, exist_ok=True)

for img_name in img_names:
    img_stem = os.path.splitext(img_name)[0]
    prob = predict_probability_mask(best_model, os.path.join(TEST_IMG_DIR, img_name))

    if prob is None:
        # No detections: export empty masks at the network resolution.
        prob = np.zeros((IMGSZ, IMGSZ), dtype=np.float32)

    prob_png = (prob * 255).astype(np.uint8)
    cv2.imwrite(os.path.join(PROBABILITY_EXPORT_DIR, f"{img_stem}_pred.png"), prob_png)

    binary_png = ((prob > PRED_THRESHOLD).astype(np.uint8) * 255)
    cv2.imwrite(os.path.join(BINARY_EXPORT_DIR, f"{img_stem}_pred.png"), binary_png)

print(f"Probability masks written to: {PROBABILITY_EXPORT_DIR}")
print(f"Binary masks written to:      {BINARY_EXPORT_DIR}")

## 6. Export validation predictions

Same convention as the test export. Used by `13_ensemble.ipynb` (ensemble
weights and scheme selection on the validation split),
`14_stacking_random_forest.ipynb` and the uniform validation-Dice table.

In [ ]:
VAL_IMG_DIR = "split_data/val/images"
VAL_PROBABILITY_EXPORT_DIR = "probability_masks/val/yolo"
VAL_BINARY_EXPORT_DIR = "val_masks/yolo"

os.makedirs(VAL_PROBABILITY_EXPORT_DIR, exist_ok=True)
os.makedirs(VAL_BINARY_EXPORT_DIR, exist_ok=True)

val_img_names = sorted([f for f in os.listdir(VAL_IMG_DIR)
                        if f.lower().endswith((".png", ".jpg", ".jpeg"))])

for img_name in val_img_names:
    img_stem = os.path.splitext(img_name)[0]
    prob = predict_probability_mask(best_model, os.path.join(VAL_IMG_DIR, img_name))

    if prob is None:
        prob = np.zeros((IMGSZ, IMGSZ), dtype=np.float32)

    cv2.imwrite(os.path.join(VAL_PROBABILITY_EXPORT_DIR, f"{img_stem}_pred.png"),
                (prob * 255).astype(np.uint8))
    cv2.imwrite(os.path.join(VAL_BINARY_EXPORT_DIR, f"{img_stem}_pred.png"),
                ((prob > PRED_THRESHOLD).astype(np.uint8) * 255))

print(f"Validation probability masks written to: {VAL_PROBABILITY_EXPORT_DIR}")
print(f"Validation binary masks written to:      {VAL_BINARY_EXPORT_DIR}")

## 7. Test-time-augmented export (optional)

Model-level TTA over six dihedral transforms of the input; predictions are
mapped back to the original orientation and averaged. Consumed by the TTA
protocol of `15_postprocessing_methods.ipynb`.

In [ ]:
TTA_MODES = ("identity", "hflip", "vflip", "rot90", "rot180", "rot270")

def tta_apply(a, mode):
    if mode == "hflip":
        return a[:, ::-1]
    if mode == "vflip":
        return a[::-1, :]
    if mode == "rot90":
        return np.rot90(a, 1)
    if mode == "rot180":
        return np.rot90(a, 2)
    if mode == "rot270":
        return np.rot90(a, 3)
    return a

def tta_invert(a, mode):
    if mode == "hflip":
        return a[:, ::-1]
    if mode == "vflip":
        return a[::-1, :]
    if mode == "rot90":
        return np.rot90(a, -1)
    if mode == "rot180":
        return np.rot90(a, -2)
    if mode == "rot270":
        return np.rot90(a, -3)
    return a

TTA_PROBABILITY_EXPORT_DIR = "probability_masks/test_tta/yolo"
os.makedirs(TTA_PROBABILITY_EXPORT_DIR, exist_ok=True)

for img_name in img_names:
    img_stem = os.path.splitext(img_name)[0]
    bgr = cv2.imread(os.path.join(TEST_IMG_DIR, img_name))

    ref = predict_probability_mask(best_model, np.ascontiguousarray(bgr))
    if ref is None:
        ref = np.zeros((IMGSZ, IMGSZ), dtype=np.float32)
    acc = ref.astype(np.float64).copy()

    for mode in TTA_MODES[1:]:
        t = np.ascontiguousarray(tta_apply(bgr, mode))
        p = predict_probability_mask(best_model, t)
        if p is None:
            continue
        p = np.ascontiguousarray(tta_invert(p, mode)).astype(np.float32)
        if p.shape != ref.shape:
            p = cv2.resize(p, (ref.shape[1], ref.shape[0]),
                           interpolation=cv2.INTER_LINEAR)
        acc += p

    prob = acc / len(TTA_MODES)
    cv2.imwrite(os.path.join(TTA_PROBABILITY_EXPORT_DIR, f"{img_stem}_pred.png"),
                (prob * 255).astype(np.uint8))

print(f"TTA probability masks written to: {TTA_PROBABILITY_EXPORT_DIR}")